In [1]:
# Importing all the libraries

import numpy as np
from scipy.stats import norm
from scipy.optimize import fsolve

#Black-Scholes Model

In [2]:
# Calculating call option price using Black-Scholes Model

sigma = 0.4
S0 = 100
K = 102.5
r = 0.01
t = 1

d1 = ((np.log(S0/K) + (r + sigma**2/2)*t) / sigma*np.sqrt(t))
d2 = d1 - sigma*np.sqrt(t)

call_price = S0*norm.cdf(d1) - K*np.exp(-r *t)*norm.cdf(d2)
call_price

np.float64(15.239829429316998)

## Defining function for Black-Scholes Model

In [3]:
def call_price(sigma, S0, K, r, t):
 d1 = ((np.log(S0/K) + (r + sigma**2/2)*t) / sigma*np.sqrt(t))
 d2 = d1 - sigma*np.sqrt(t)

 C = S0*norm.cdf(d1) - K*np.exp(-r *t)*norm.cdf(d2)
 return C

In [4]:
# Calculate call price using function defined above
call_price(0.4 ,100, 102.5, 0.01, 1)

np.float64(15.239829429316998)

# Binomial Model

###2 Step Binomial Tree

In [5]:
# Our initial stock price
S0 = 100

# Time to expiration in years
t = 1

# Assumed up and down percentages
u = 1.05
d = 1/u

# Probability of an up move
q = (np.exp(r * t)- d) / (u-d)
print("Probablity of an upmove(q) = ",q)

# Stock prices at expiration
S_u = u * S0
S_d = d * S0

# Option strike price and risk-free rate
K = 102.5
r = 0.01

# Call price at up and down nodes
C_u = S_u - K
C_d = 0

# Print out the stock prices at expiration
print("Stock price at upnode(S_u) = ",S_u,", Stock price at downnode(S_u) = ",S_d)

# Calculate and print the call price at t = 0
C = np.exp(-r * t) * (q*C_u + (1-q)*C_d)
print("Call option price at time 0 = ",C)

Probablity of an upmove(q) =  0.5907578091548912
Stock price at upnode(S_u) =  105.0 , Stock price at downnode(S_u) =  95.23809523809523
Call option price at time 0 =  1.462199176849557


### Multi Step Binomial Tree

Step 1. Set the value of input parameters

In [6]:
# Define the number of layers
N = 100

# Time to expiration, the we rescale it to time per layer step
t = 1
dt = t / (N - 1)

# Iniital stock price, strike price, and risk-free rate
S0 = 100
K = 102.5
r = 0.01

# Assume a volatility and calculate the size of an up move, down move, and probability
sigma = 0.4
u = np.exp(sigma * np.sqrt(dt))
d = 1/u
q = (np.exp(r * dt) - d) / (u - d)

print(u)
print(q)

1.0410205318220893
0.4912069975856682


#### Step 2. Create square matrices to hold stocks and European call option prices

In [7]:
# Create some empty matrices to hold our stock and call prices.
stock_prices = np.zeros((N, N))
call_prices = np.zeros((N, N))

#### Step 3. Populate stock prices matrix

In [8]:
# Put our initial price in the matrix
stock_prices[0,0] = S0

# Fill out the remaining values
for i in range(1, N):
    M = i + 1
    stock_prices[i, 0] = d*stock_prices[i-1, 0]
    for j in range(1, M):
        stock_prices[i, j] = u*stock_prices[i-1, j-1] #i denotes time step, j denotes height of binomial tree

In [9]:
stock_prices

array([[1.00000000e+02, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [9.60595847e+01, 1.04102053e+02, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [9.22744382e+01, 1.00000000e+02, 1.08372375e+02, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       ...,
       [2.02510887e+00, 2.19465858e+00, 2.37840362e+00, ...,
        4.93800612e+03, 0.00000000e+00, 0.00000000e+00],
       [1.94531117e+00, 2.10817991e+00, 2.28468464e+00, ...,
        4.74342817e+03, 5.14056576e+03, 0.00000000e+00],
       [1.86865783e+00, 2.02510887e+00, 2.19465858e+00, ...,
        4.55651741e+03, 4.93800612e+03, 5.35143450e+03]])

#### Step 4. Populate call option prices matrix

In [10]:
#  Calculate the call price at expiration. If the call price is less than zero, it is out-of-the-money so we replace those values with zero.
expiration = stock_prices[-1,:] - K #Taking last row and all the columns
expiration.shape = (expiration.size, )
expiration = np.where(expiration >= 0, expiration, 0)

#  Set the last row of the call matrix to our expiration values
call_prices[-1,:] =  expiration

#### Step 5. Backpropogate call prices to fill the tree

In [11]:
#  Backpropagate to fill out our tree
for i in range(N - 2,-1,-1):
    for j in range(i + 1):
        call_prices[i,j] = np.exp(-r * dt) * ((1-q) * call_prices[i+1,j] + q * call_prices[i+1,j+1])

In [12]:
print(call_prices[0,0])

15.249020281536971


In [13]:
#As we increase number of steps(N) the binomial call option price converges to black scholes call option price
#As N increases dt becomes smaller binomial model becomes more continuous and approaches Black-Scholes Model

#### Defining function for Multi-Level Binomial Tree

In [14]:
def binomial(sigma, S, K, r, t, N, option_type = 'C'):

    t = 1; t = t/(N-1)
    S0 = S

    u =  np.exp(sigma * np.sqrt(t))
    d = 1/u
    p = (np.exp(r * t) - d) / (u - d)

    stock_prices  = np.zeros((N, N))
    option_prices = np.zeros((N, N))

    stock_prices[0, 0] = S0

    #  Fill out the remaining values
    for i in range(1, N):
        M = i + 1
        stock_prices[i, 0] = d * stock_prices[i-1, 0]
        for j in range(1, M ):
            stock_prices[i, j] = u * stock_prices[i - 1, j - 1]

    #  Calculate the option price at expiration.  if the call price is less than zero, it is out-of-the-money so we replace those values with zero.
    if option_type == 'C':
        expiration = stock_prices[-1,:] - K
    else:
        expiration = K - stock_prices[-1,:]

    expiration.shape = (expiration.size, )
    expiration = np.where(expiration >= 0, expiration, 0)

    #  Set the last row of the call matrix to our expiration values
    option_prices[-1,:] =  expiration

    #  Backpropagate to fill out our tree
    for i in range(N - 2,-1,-1):
        for j in range(i + 1):
            option_prices[i,j] = np.exp(-r * t) * ((1-p) * option_prices[i+1,j] + p * option_prices[i+1,j+1])

    return option_prices[0,0]

In [15]:
#Checking above defined function
binomial(0.4, 100, 102.5, 0.01, 1, 100 , 'C')


np.float64(15.249020281536971)

# Implied Volatility Calculation

In [16]:
# For implied volatility both black scholes and binomial model can be used
# This is our objective function
def objective(sigma, S, K, r, t, N, C0):
  res = binomial(sigma, S, K, r, t, N) - C0
  return res

In [17]:
#  Parameters from above.
N = 100
t = 1
S0 = 100
K = 102.5
r = 0.01

#  We'll use the option price obtained from market
C0 = 15.3

#  Wrap the addition arguments in a tuple
additional_arguments = (S0, K, r, t, N, C0)

#  Solve the problem via fsolve
implied_volatility = fsolve(objective, 0.25, args = additional_arguments)

/tmp/ipykernel_7459/16211066.py:18: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  stock_prices[i, 0] = d * stock_prices[i-1, 0]
/tmp/ipykernel_7459/16211066.py:20: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  stock_prices[i, j] = u * stock_prices[i - 1, j - 1]
/tmp/ipykernel_7459/16211066.py:37: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  option_prices[i,j] = np.exp(-r * t) * ((1-p) * option_prices[i+1,j] + p * option_prices[i+1,j+1])


In [18]:
print(implied_volatility)

[0.40128954]


# Key Findings

-  It was observed that option prices increased significantly with higher volatility and longer time to maturity.

- The Binomial Option Pricing Model provided greater flexibility compared to Black-Scholes by modelling stock prices through discrete upward and downward movements across multiple time steps.

- As the number of steps in the Binomial Model increased, the option prices gradually converged toward the Black-Scholes prices, validating the theoretical relationship between the two models.

- Implied Volatility estimation demonstrated how market prices can be used to reverse-engineer the volatility expected by market participants rather than relying on historical volatility alone.

- It was observed that even small changes in implied volatility produced noticeable changes in option prices, highlighting volatility as one of the most critical components in derivatives pricing.

- The project also showed that market option prices often differ from theoretical prices due to changing market expectations, volatility fluctuations, and real-world trading conditions.
